# Detector de objetos com YOLO11n (para o site de assistência)

Mesmo dataset do notebook do SSDLite (`dataset_open_images.zip` no Drive, os 49 objetos do Open Images),
treinado com o **YOLO11n** (Ultralytics). Com os mesmos dados, o YOLO11n costuma ter mAP bem maior que o
SSDLite MobileNetV3 e roda com velocidade parecida no celular.

**Antes:** rode pelo menos uma vez o notebook `treino_detector_mobilenet.ipynb` até a parte 2, para o zip com as fotos
existir no Drive. Este notebook só reaproveita esse zip.

**Licença:** o Ultralytics é AGPL-3.0. Para um site com código aberto (como o seu repositório público) e uso acadêmico,
não há problema; para um produto fechado, seria preciso a licença comercial deles.

Tempo: com GPU T4 e 320 px, cerca de 1 a 1,5 min por época com ~15 mil fotos. 80 épocas ≈ 1h30 a 2h
(para antes se parar de melhorar por 15 épocas).

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics onnx onnxruntime onnxslim

## 1. Configuração

In [ ]:
PASTA_DRIVE = "/content/drive/MyDrive/Mestrado/detector_assistencia"
# Qual zip usar: "dataset_open_images.zip" (rodada normal) ou "dataset_open_images_foco.zip" (mais fotos dos objetos fracos)
ZIP = "dataset_open_images_foco.zip"

MODELO_BASE = "yolo11n.pt"     # pré-treinado no COCO; "yolo11s.pt" é ~3x mais pesado e mais preciso
TAMANHO = 320                  # entrada no celular; 416 acha objetos pequenos melhor, mas é ~70% mais lento
EPOCAS = 80
PACIENCIA = 15                 # para se o mAP de validação não melhorar por tantas épocas
LOTE = 64
SEMENTE = 0

## 2. Drive e conversão para o formato YOLO

In [ ]:
import os, json, shutil, zipfile
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

zip_drive = Path(PASTA_DRIVE) / ZIP
if not zip_drive.exists():
    alternativo = Path(PASTA_DRIVE) / "dataset_open_images.zip"
    assert alternativo.exists(), "Rode antes o notebook do SSDLite até a parte 2, para criar o zip com as fotos."
    print(f"{ZIP} não existe; usando {alternativo.name}")
    zip_drive = alternativo
COCO_DIR = Path("/content/dataset")
if not COCO_DIR.exists():
    zipfile.ZipFile(zip_drive).extractall(COCO_DIR)

ler = lambda split: json.loads((COCO_DIR / split / "_annotations.coco.json").read_text())
# Mesma ordem de classes do notebook do SSDLite (só categorias usadas, em ordem alfabética),
# para o objetos_labels.txt ser o mesmo
usadas = set()
for split in ("train", "valid", "test"):
    usadas |= {a["category_id"] for a in ler(split)["annotations"]}
categorias = sorted((c for c in ler("train")["categories"] if c["id"] in usadas), key=lambda c: c["name"])
CLASSES = [c["name"] for c in categorias]
indice = {c["id"]: i for i, c in enumerate(categorias)}   # YOLO numera de 0

YOLO_DIR = Path("/content/yolo")
for split in ("train", "valid", "test"):
    coco = ler(split)
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
    caixas = {}
    for a in coco["annotations"]:
        caixas.setdefault(a["image_id"], []).append(a)
    for img in coco["images"]:
        origem = COCO_DIR / split / img["file_name"]
        if not origem.exists():
            continue
        shutil.move(origem, YOLO_DIR / "images" / split / img["file_name"])
        w, h = img["width"], img["height"]
        linhas = []
        for a in caixas.get(img["id"], []):
            x, y, bw, bh = a["bbox"]
            linhas.append(f"{indice[a['category_id']]} {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}")
        (YOLO_DIR / "labels" / split / (Path(img["file_name"]).stem + ".txt")).write_text("\n".join(linhas))
    print(split, len(coco["images"]), "fotos")

(YOLO_DIR / "dados.yaml").write_text(
    f"path: {YOLO_DIR}\ntrain: images/train\nval: images/valid\ntest: images/test\n"
    + "names:\n" + "".join(f"  {i}: {json.dumps(n, ensure_ascii=False)}\n" for i, n in enumerate(CLASSES)))
print(len(CLASSES), "classes:", CLASSES)

## 3. Treino

In [ ]:
from ultralytics import YOLO

PASTA_SAIDA = Path(PASTA_DRIVE) / "yolo11n"
modelo = YOLO(MODELO_BASE)
modelo.train(
    data=str(YOLO_DIR / "dados.yaml"), imgsz=TAMANHO, epochs=EPOCAS, patience=PACIENCIA, batch=LOTE,
    workers=os.cpu_count(), seed=SEMENTE, cos_lr=True,
    project=str(PASTA_SAIDA), name="treino", exist_ok=True,   # pesos e gráficos vão direto para o Drive
)

Se o Colab desconectar no meio, rode a célula abaixo em vez da de cima: ela continua de onde parou
(o `last.pt` fica salvo no Drive a cada época).

In [ ]:
# modelo = YOLO(str(PASTA_SAIDA / "treino" / "weights" / "last.pt")); modelo.train(resume=True)

## 4. Resultado no teste (por objeto), comparado com o SSDLite

In [ ]:
import pandas as pd

melhor = YOLO(str(PASTA_SAIDA / "treino" / "weights" / "best.pt"))
r = melhor.val(data=str(YOLO_DIR / "dados.yaml"), split="test", imgsz=TAMANHO, batch=LOTE, plots=False)
print(f"YOLO11n no teste: mAP50-95 = {r.box.map:.3f} | mAP50 = {r.box.map50:.3f}")

tabela = pd.DataFrame({"objeto": [CLASSES[i] for i in r.box.ap_class_index], "AP_yolo": r.box.maps[r.box.ap_class_index]})
ssd = Path(PASTA_DRIVE) / "ap_por_objeto.csv"         # da rodada normal do SSDLite, se existir
if ssd.exists():
    anterior = pd.read_csv(ssd)
    col = [c for c in anterior.columns if c != "objeto"][0]
    tabela = tabela.merge(anterior.rename(columns={col: "AP_ssdlite"}), on="objeto", how="left")
tabela = tabela.sort_values("AP_yolo")
tabela.to_csv(PASTA_SAIDA / "ap_por_objeto_yolo.csv", index=False)
tabela.round(3)

## 5. Exportar para o site

Gera `objetos.onnx` (com NMS incluído: até 100 caixas `[x0, y0, x1, y1, pontuação, classe]` em pixels da entrada)
e `objetos_labels.txt` em `yolo11n/modelo_site/`. Mande os dois para mim (ou copie para `teste-mestrado/modelos/`):
o site reconhece sozinho que é um modelo YOLO pela saída.

In [ ]:
import numpy as np, onnxruntime as ort

saida = PASTA_SAIDA / "modelo_site"
saida.mkdir(exist_ok=True)
try:
    arquivo = melhor.export(format="onnx", imgsz=TAMANHO, opset=17, simplify=True, nms=True, conf=0.25, max_det=100)
except TypeError:   # versões antigas do Ultralytics não têm nms=True; o site faz o NMS sozinho nesse caso
    arquivo = melhor.export(format="onnx", imgsz=TAMANHO, opset=17, simplify=True)
shutil.copy(arquivo, saida / "objetos.onnx")
(saida / "objetos_labels.txt").write_text("\n".join(CLASSES) + "\n", encoding="utf-8")

# Confere o ONNX numa foto do teste
sessao = ort.InferenceSession(str(saida / "objetos.onnx"))
print("entrada:", sessao.get_inputs()[0].shape, "| saídas:", [(o.name, o.shape) for o in sessao.get_outputs()])
foto = next((YOLO_DIR / "images" / "test").iterdir())
ref = melhor.predict(str(foto), imgsz=TAMANHO, conf=0.25, verbose=False)[0]
print("caixas no PyTorch:", len(ref.boxes))
print("tamanho:", round((saida / "objetos.onnx").stat().st_size / 1e6, 1), "MB ->", saida)